In [14]:
import json
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
import vector
import pandas as pd
import torch
import uproot

In [5]:
def to_p4(p4_obj):
    return vector.awk(
        ak.zip(
            {
                "mass": p4_obj.tau,
                "x": p4_obj.x,
                "y": p4_obj.y,
                "z": p4_obj.z,
            }
        )
    )

In [6]:
z_train = ak.from_parquet("/scratch/persistent/norman/ml-tau/cld/v1.2.5_key4hep_2025-05-29/260325/z_test.parquet")

In [7]:
mask = to_p4(z_train.reco_jet_p4s).pt > 5
data_ptcut = z_train[mask]

In [11]:
chargearray = ak.flatten(data_ptcut["gen_jet_tau_charge"], axis = -1)
unique, counts = np.unique(chargearray, return_counts=True)
for u, c in sorted(zip(unique, counts), key=lambda x: -x[1]):
    print(f"charge recojet pt>5 cut {u:>4}: {c}")

charge recojet pt>5 cut  1.0: 60591.0
charge recojet pt>5 cut -1.0: 60290.0


In [12]:
chargearray_0 = ak.flatten(z_train["gen_jet_tau_charge"], axis = -1)
unique, counts = np.unique(chargearray_0, return_counts=True)
for u, c in sorted(zip(unique, counts), key=lambda x: -x[1]):
    print(f"charge no cuts {u:>4}: {c}")

charge no cuts  1.0: 64239.0
charge no cuts -1.0: 64146.0


In [15]:
file = uproot.open("/local/joosep/mlpf/cld/v1.2.5_key4hep_2025-05-29/gen/p8_ee_Z_tautau_ecm91/root/reco_p8_ee_Z_tautau_ecm91_707040.root")
print(file.keys())

['events;2', 'events;1', 'configuration_metadata;1', 'metadata;1', 'podio_metadata;1']


In [16]:
tree = file["events"].arrays()